# Comparing `suite2p` and `normcorre` registration

`photon-mosaic` corrects motion in two steps: it estimates how each frame moved, then it applies that motion to the frames as they are read. Two backends can do both steps, `suite2p` and `normcorre` (CaImAn's NoRMCorre). When their registered movies differ, the difference could come from either step. Here we separate the two steps: we apply each backend's motion with each backend, four movies in all, and score every frame of every movie.

This shows what the functions do, on 200 frames of `suite2p`'s sample data. It is not a benchmark: which backend registers your data better depends on the data and the settings.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from photon_mosaic.preprocessing import compute_motion, frame_correlations, register_motion
from photon_mosaic.sample_data import download_suite2p_google_drive

## The movie

One file of `suite2p`'s sample data: one plane, 200 frames, recorded at 30 Hz.

In [ ]:
imaging = download_suite2p_google_drive(1)
print(imaging)

## Estimating the motion with `suite2p`

`compute_motion` estimates how far every frame moved relative to a reference image. With its default settings `suite2p` does it in two passes: one shift for the whole frame (rigid), then one shift per block of the frame (non-rigid). The rigid shifts are in `motion.displacements`: one array per epoch, `(frames, planes, 2)`, in pixels along `(y, x)`.

In [ ]:
suite2p_motion = compute_motion(imaging=imaging, method="suite2p")

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 4))
for i, axis in enumerate("yx"):
    axes[i].plot(suite2p_motion.displacements[0][:, 0, i], label="suite2p")
    axes[i].set_ylabel(f"{axis} shift (px)")
axes[0].legend()
axes[-1].set_xlabel("frame")

## `suite2p`'s reference image

To estimate the motion, `suite2p` first builds a sharp reference image: it picks the frames that correlate best with each other, averages them, and refines that average by registering the frames to it. It keeps the result in `motion.reference`, one image per plane.

In [ ]:
reference = suite2p_motion.reference[0]

plt.figure(figsize=(6, 5))
plt.imshow(reference, cmap="gray")
plt.title("suite2p reference image")

## The score

A sample that moves smears its frames relative to one another, so a frame of a moving movie matches a sharp reference only in part. `frame_correlations` scores a movie by the correlation of every frame with a template: the better the movie is registered, the higher the score. We use `suite2p`'s reference as the one template for every movie, so the scores compare; since it is `suite2p`'s own image, it may favour `suite2p` a little. On the raw movie it gives the baseline.

In [ ]:
raw_scores = frame_correlations(imaging, template=reference)

plt.figure(figsize=(8, 3))
plt.plot(raw_scores, label="raw")
plt.xlabel("frame")
plt.ylabel("correlation with the reference")
plt.legend()

## A second estimate, with `normcorre`

The same call with `method="normcorre"` estimates the motion again, independently, also rigid then non-rigid (per overlapping patch). Its shifts follow the same convention, so they go on the same axes. `suite2p`'s rigid shifts are whole pixels; `normcorre`'s are not.

In [ ]:
normcorre_motion = compute_motion(imaging=imaging, method="normcorre")

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 4))
for i, axis in enumerate("yx"):
    axes[i].plot(suite2p_motion.displacements[0][:, 0, i], label="suite2p")
    axes[i].plot(normcorre_motion.displacements[0][:, 0, i], label="normcorre")
    axes[i].set_ylabel(f"{axis} shift (px)")
axes[0].legend()
axes[-1].set_xlabel("frame")

## The motion across the field of view

The non-rigid pass gives one shift per patch of the frame. Both backends can describe their motion the same way, with `motion.patch_motion(epoch, plane)`: the shift of every patch in every frame, `(frames, rows of patches, columns of patches, 2)`, rigid part included, and where the patches are centred. Here it is for the frame where `suite2p` saw the largest shift, as arrows at the patch centres, drawn 30 times longer than the shift.

In [ ]:
frame = int(np.abs(suite2p_motion.displacements[0][:, 0]).sum(axis=1).argmax())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

suite2p_shifts, (suite2p_cy, suite2p_cx) = suite2p_motion.patch_motion(0, 0)
x, y = np.meshgrid(suite2p_cx, suite2p_cy)
axes[0].imshow(reference, cmap="gray")
axes[0].quiver(x, y, suite2p_shifts[frame, ..., 1], suite2p_shifts[frame, ..., 0],
               color="tab:blue", angles="xy", scale_units="xy", scale=1 / 30)
axes[0].set_title(f"suite2p, frame {frame}")

normcorre_shifts, (normcorre_cy, normcorre_cx) = normcorre_motion.patch_motion(0, 0)
x, y = np.meshgrid(normcorre_cx, normcorre_cy)
axes[1].imshow(reference, cmap="gray")
axes[1].quiver(x, y, normcorre_shifts[frame, ..., 1], normcorre_shifts[frame, ..., 0],
               color="tab:orange", angles="xy", scale_units="xy", scale=1 / 30)
axes[1].set_title(f"normcorre, frame {frame}")

## The border shifting leaves

Shifting a frame 3 pixels down leaves a 3-pixel strip at the top with no data, which gets filled in. Those filled-in pixels would distort the score, so we leave them out. Each motion records how wide its strips are, from its largest shift, as the rows and columns that stay real data: `yranges` and `xranges`, per epoch and plane. We keep the rectangle that is real data for both motions, and score every movie on it, the raw one too, so all of them are scored on the same pixels.

In [ ]:
y0 = max(suite2p_motion.yranges[0][0][0], normcorre_motion.yranges[0][0][0])
y1 = min(suite2p_motion.yranges[0][0][1], normcorre_motion.yranges[0][0][1])
x0 = max(suite2p_motion.xranges[0][0][0], normcorre_motion.xranges[0][0][0])
x1 = min(suite2p_motion.xranges[0][0][1], normcorre_motion.xranges[0][0][1])
bounds = ((y0, y1), (x0, x1))
print("rows", bounds[0], "columns", bounds[1])

## Applying a motion

`register_motion` applies a motion lazily: frames are corrected only when they are read. By default the backend that estimated the motion applies it.

In [ ]:
suite2p_by_suite2p = register_motion(imaging=imaging, motion=suite2p_motion)

raw_scores = frame_correlations(imaging, bounds=bounds, template=reference)
suite2p_by_suite2p_scores = frame_correlations(suite2p_by_suite2p, bounds=bounds, template=reference)

plt.figure(figsize=(8, 3))
plt.plot(raw_scores, label="raw")
plt.plot(suite2p_by_suite2p_scores, label="suite2p motion, applied by suite2p")
plt.xlabel("frame")
plt.ylabel("correlation with the reference")
plt.legend()

## Applying a motion with the other backend

`method` chooses the backend that applies the motion. Because both motions can describe themselves with `patch_motion`, `normcorre` can apply `suite2p`'s motion, and the other way round. Neither `suite2p` nor CaImAn can do this on their own; the conversion is `photon-mosaic`'s.

In [ ]:
suite2p_by_normcorre = register_motion(imaging=imaging, motion=suite2p_motion, method="normcorre")

suite2p_by_normcorre_scores = frame_correlations(suite2p_by_normcorre, bounds=bounds, template=reference)

plt.figure(figsize=(8, 3))
plt.plot(raw_scores, label="raw")
plt.plot(suite2p_by_suite2p_scores, label="suite2p motion, applied by suite2p")
plt.plot(suite2p_by_normcorre_scores, label="suite2p motion, applied by normcorre")
plt.xlabel("frame")
plt.ylabel("correlation with the reference")
plt.legend()

## All four

The same for `normcorre`'s motion gives the other two movies.

In [ ]:
normcorre_by_normcorre = register_motion(imaging=imaging, motion=normcorre_motion)
normcorre_by_suite2p = register_motion(imaging=imaging, motion=normcorre_motion, method="suite2p")

normcorre_by_normcorre_scores = frame_correlations(normcorre_by_normcorre, bounds=bounds, template=reference)
normcorre_by_suite2p_scores = frame_correlations(normcorre_by_suite2p, bounds=bounds, template=reference)

plt.figure(figsize=(8, 4))
plt.plot(raw_scores, label=f"raw ({raw_scores.mean():.3f})")
plt.plot(suite2p_by_suite2p_scores, label=f"suite2p motion, by suite2p ({suite2p_by_suite2p_scores.mean():.3f})")
plt.plot(suite2p_by_normcorre_scores, label=f"suite2p motion, by normcorre ({suite2p_by_normcorre_scores.mean():.3f})")
plt.plot(normcorre_by_normcorre_scores, label=f"normcorre motion, by normcorre ({normcorre_by_normcorre_scores.mean():.3f})")
plt.plot(normcorre_by_suite2p_scores, label=f"normcorre motion, by suite2p ({normcorre_by_suite2p_scores.mean():.3f})")
plt.xlabel("frame")
plt.ylabel("correlation with the reference")
plt.legend(fontsize="small")